# 01 — Data Preparation

This notebook:
1. Defines all project parameters in one place (`CONFIG`)
2. Loads BGT and BRT shapefiles for Arnhem, Wageningen, Apeldoorn
3. Assigns cities to train / val / test splits
4. Encodes categorical attributes to integers
5. Tiles each city into fixed-size grid cells
6. Saves processed tiles to disk for use in later notebooks

**City assignment:**
- Train → Apeldoorn (largest, most variety)
- Val   → Arnhem
- Test  → Wageningen


## 0. Install dependencies (Colab)

In [ ]:
# Run this cell only on Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install geopandas shapely fiona pyproj --quiet

## 1. CONFIG — edit all parameters here

In [4]:
from pathlib import Path

# ─────────────────────────────────────────────
# PROJECT CONFIG  — change values here only
# ─────────────────────────────────────────────
CONFIG = {

    # ── Paths ──────────────────────────────────
    # Root folder that contains your data.
    # On Colab you can mount Google Drive and point here.
    # Expected structure:
    #   data_root/
    #     arnhem/bgt/    ← BGT shapefiles for Arnhem
    #     arnhem/brt/    ← BRT shapefiles for Arnhem
    #     wageningen/bgt/
    #     wageningen/brt/
    #     apeldoorn/bgt/
    #     apeldoorn/brt/
    "data_root":    Path("data"),
    "output_root":  Path("processed"),   # where prepared tiles are saved

    # ── City → split assignment ─────────────────
    # Change these if you want a different city per split
    "split_map": {
        "apeldoorn":  "train",
        "arnhem":     "val",
        "wageningen": "test",
    },

    # ── Coordinate reference system ─────────────
    # RD New is standard for Dutch datasets
    "crs": "EPSG:28992",

    # ── Tiling ──────────────────────────────────
    # Side length of each square tile in metres (RD New units)
    "tile_size_m": 500,

    # Minimum fraction of a tile that must be covered by data
    # before the tile is kept (avoids empty edge tiles)
    "tile_min_coverage": 0.3,

    # ── BGT layer files ─────────────────────────
    # List the GML (or SHP) filenames to load from each city's bgt/ folder.
    # For GML: place the matching .gfs file in the same folder — GDAL picks it up automatically.
    # BGT GML column names often use dashes e.g. 'plus-type' — check with the
    # column inspection cell after loading if encoding gives 0 classes.
    "bgt_layers": [
        "bgt_wegdeel.gml",
        # "onbegroeidterreindeel.gml",
        # "begroeidterreindeel.gml",
        # "waterdeel.gml",
        # "pand.gml",
        # "scheiding.gml",
    ],

    # ── BRT layer files ─────────────────────────
    "brt_layers": [
        "top10nl_wegdeel.gml",
        # "terrein.gml",
        # "waterdeel.gml",
        # "gebouw.gml",
    ],

    # ── Attribute encoding ───────────────────────
    # Column name in BGT/BRT that holds the feature type/class.
    # Adjust to match your actual attribute names.
    "bgt_class_col": "plus_type",    # or 'bgt_type', 'typeweg', etc.
    "brt_class_col": "typewegde",    # or 'typegebou', 'typewater', etc.

    # Unknown / missing class label (used when a value is not in the encoder)
    "unknown_class": "__UNKNOWN__",
}

# Derived paths (do not edit — computed from CONFIG above)
CONFIG["output_root"].mkdir(parents=True, exist_ok=True)
for split in ["train", "val", "test"]:
    (CONFIG["output_root"] / split).mkdir(parents=True, exist_ok=True)

print("CONFIG loaded.")
print(f"  data_root   : {CONFIG['data_root'].resolve()}")
print(f"  output_root : {CONFIG['output_root'].resolve()}")
print(f"  splits      : {CONFIG['split_map']}")

CONFIG loaded.
  data_root   : C:\Github\GNN-Map-generalization-BGT-to-BRT\data
  output_root : C:\Github\GNN-Map-generalization-BGT-to-BRT\processed
  splits      : {'apeldoorn': 'train', 'arnhem': 'val', 'wageningen': 'test'}


## 2. Imports

In [5]:
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import box
from shapely.errors import ShapelyDeprecationWarning

warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

print("Imports OK")

Imports OK


## 3. Load BGT and BRT data

Supports both `.gml` and `.shp` files — set the extensions in CONFIG.
For GML files, place the `.gfs` file in the same folder as the `.gml` file
and GDAL will pick it up automatically. No code change needed for `.gfs`.

In [6]:
def load_layers(city: str, dataset: str, layer_files: list, config: dict) -> gpd.GeoDataFrame:
    """
    Load and concatenate layers for a given city and dataset (bgt or brt).
    Supports both .gml and .shp files — just set the extension in CONFIG.

    GML notes:
    - Place the .gfs file in the same folder as the .gml file.
      GDAL picks it up automatically — no extra code needed.
    - GML files can contain multiple feature layers internally.
      This function reads all of them and keeps only Polygon/MultiPolygon types.
    - BGT GML column names often use dashes (e.g. 'plus-type') — check with
      the column inspection cell below if encoding gives 0 classes.
    """
    import fiona
    folder = config["data_root"] / city / dataset
    frames = []

    for layer_file in layer_files:
        path = folder / layer_file
        if not path.exists():
            print(f"  [SKIP] {path} not found")
            continue

        # GML files may contain multiple named layers; SHP always has one.
        # fiona.listlayers() returns the list for both formats.
        try:
            available_layers = fiona.listlayers(str(path))
        except Exception as e:
            print(f"  [WARN] Could not inspect layers in {path.name}: {e}")
            available_layers = [None]  # fall back to default layer

        for layer_name in available_layers:
            try:
                if layer_name is not None:
                    gdf = gpd.read_file(path, layer=layer_name)
                else:
                    gdf = gpd.read_file(path)
            except Exception as e:
                print(f"  [WARN] Could not read layer '{layer_name}' from {path.name}: {e}")
                continue

            if len(gdf) == 0:
                continue

            # Strip extension for source tracking, append layer name if GML
            ext   = path.suffix
            label = path.stem + (f"_{layer_name}" if ext == ".gml" and layer_name else "")
            gdf["source_layer"] = label
            gdf["city"]         = city

            # Reproject if needed
            if gdf.crs is None:
                print(f"  [WARN] {path.name} has no CRS — assuming {config['crs']}")
                gdf = gdf.set_crs(config["crs"])
            else:
                gdf = gdf.to_crs(config["crs"])

            # Keep only polygon geometries
            gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
            gdf = gdf[gdf.geometry.is_valid].copy()

            if len(gdf) == 0:
                continue

            frames.append(gdf)
            suffix = f" [{layer_name}]" if layer_name else ""
            print(f"  Loaded {len(gdf):,} polygons from {path.name}{suffix}")

    if not frames:
        raise FileNotFoundError(f"No {dataset} layers found for {city} in {folder}")

    combined = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=config["crs"])
    print(f"  → {city} {dataset.upper()} total: {len(combined):,} polygons\n")
    return combined


# Load all cities
bgt_data = {}
brt_data = {}

for city in CONFIG["split_map"]:
    print(f"Loading {city.upper()} BGT...")
    bgt_data[city] = load_layers(city, "bgt", CONFIG["bgt_layers"], CONFIG)

    print(f"Loading {city.upper()} BRT...")
    brt_data[city] = load_layers(city, "brt", CONFIG["brt_layers"], CONFIG)

print("All data loaded.")

Loading APELDOORN BGT...
  [SKIP] data\apeldoorn\bgt\bgt_wegdeel.gml not found


FileNotFoundError: No bgt layers found for apeldoorn in data\apeldoorn\bgt

In [ ]:
# ── Column inspection — run this before setting bgt_class_col / brt_class_col ──
# GML files from the Kadaster often use dashes in column names (e.g. 'plus-type')
# which differ from what you might expect. Check here and update CONFIG above.
train_city = [c for c, s in CONFIG["split_map"].items() if s == "train"][0]

print(f"BGT columns for {train_city}:")
print(" ", [c for c in bgt_data[train_city].columns if c not in ["geometry", "city", "source_layer"]])

print(f"\nBRT columns for {train_city}:")
print(" ", [c for c in brt_data[train_city].columns if c not in ["geometry", "city", "source_layer"]])

print(f"\nCurrent CONFIG settings:")
print(f"  bgt_class_col = '{CONFIG['bgt_class_col']}'")
print(f"  brt_class_col = '{CONFIG['brt_class_col']}'")

bgt_ok = CONFIG["bgt_class_col"] in bgt_data[train_city].columns
brt_ok = CONFIG["brt_class_col"] in brt_data[train_city].columns
print(f"\n  bgt_class_col found in data: {'✓' if bgt_ok else '✗ UPDATE CONFIG'}")
print(f"  brt_class_col found in data: {'✓' if brt_ok else '✗ UPDATE CONFIG'}")

## 4. Encode categorical class labels

We build a label encoder from the **training city only** (Apeldoorn),
then apply it to val and test. This prevents data leakage.
Unknown classes in val/test get the `__UNKNOWN__` code.

In [ ]:
class LabelEncoder:
    """
    Simple string → integer encoder.
    Built from training data only. Handles unseen labels gracefully.
    Saved as plain JSON so it can be loaded in any notebook without pickle.
    """

    def __init__(self, unknown_label: str = "__UNKNOWN__"):
        self.unknown_label = unknown_label
        self.classes_      = []   # list of string class names
        self.label_to_int  = {}   # string → int
        self.int_to_label  = {}   # int → string
        self._fitted       = False

    def fit(self, series: pd.Series):
        """Build encoder from a pandas Series of string labels."""
        unique = sorted(series.dropna().unique().tolist())
        self.classes_     = [self.unknown_label] + unique
        self.label_to_int = {c: i for i, c in enumerate(self.classes_)}
        self.int_to_label = {i: c for i, c in enumerate(self.classes_)}
        self._fitted      = True
        return self

    def transform(self, series: pd.Series) -> pd.Series:
        """Encode a Series. Unknown labels map to 0."""
        assert self._fitted, "Call fit() first"
        return series.map(lambda x: self.label_to_int.get(x, 0)).astype(int)

    def inverse_transform(self, series: pd.Series) -> pd.Series:
        """Decode integers back to string labels."""
        assert self._fitted, "Call fit() first"
        return series.map(lambda x: self.int_to_label.get(x, self.unknown_label))

    def save(self, path):
        """Save encoder as plain JSON — no pickle, no class dependency on load."""
        data = {
            "unknown_label": self.unknown_label,
            "classes":       self.classes_,
            "label_to_int":  self.label_to_int,
            # JSON keys must be strings, so int keys become strings here
            "int_to_label":  {str(k): v for k, v in self.int_to_label.items()},
        }
        with open(path, "w") as f:
            json.dump(data, f, indent=2)

    def __repr__(self):
        return f"LabelEncoder({len(self.classes_)} classes, fitted={self._fitted})"


def load_encoder(path) -> LabelEncoder:
    """Load a LabelEncoder from a JSON file — no pickle needed."""
    with open(path) as f:
        data = json.load(f)
    enc = LabelEncoder(unknown_label=data["unknown_label"])
    enc.classes_     = data["classes"]
    enc.label_to_int = data["label_to_int"]
    enc.int_to_label = {int(k): v for k, v in data["int_to_label"].items()}
    enc._fitted      = True
    return enc


# Find the training city
train_city = [c for c, s in CONFIG["split_map"].items() if s == "train"][0]
print(f"Building encoders from training city: {train_city}")

# Build BGT encoder
bgt_encoder = LabelEncoder(unknown_label=CONFIG["unknown_class"])
bgt_col = CONFIG["bgt_class_col"]

if bgt_col in bgt_data[train_city].columns:
    bgt_encoder.fit(bgt_data[train_city][bgt_col].fillna(CONFIG["unknown_class"]))
    print(f"  BGT encoder: {len(bgt_encoder.classes_)} classes")
else:
    print(f"  [WARN] BGT class column '{bgt_col}' not found. Available: {list(bgt_data[train_city].columns)}")

# Build BRT encoder
brt_encoder = LabelEncoder(unknown_label=CONFIG["unknown_class"])
brt_col = CONFIG["brt_class_col"]

if brt_col in brt_data[train_city].columns:
    brt_encoder.fit(brt_data[train_city][brt_col].fillna(CONFIG["unknown_class"]))
    print(f"  BRT encoder: {len(brt_encoder.classes_)} classes")
else:
    print(f"  [WARN] BRT class column '{brt_col}' not found. Available: {list(brt_data[train_city].columns)}")

# Save as JSON (not pickle) so any notebook can load them without class dependencies
bgt_encoder.save(CONFIG["output_root"] / "bgt_encoder.json")
brt_encoder.save(CONFIG["output_root"] / "brt_encoder.json")
print("\nEncoders saved as JSON.")

In [ ]:
def apply_encoding(gdf: gpd.GeoDataFrame, col: str, encoder: LabelEncoder) -> gpd.GeoDataFrame:
    """
    Add an integer-encoded column '<col>_encoded' to gdf.
    Fills NaN values with the unknown label before encoding.
    """
    gdf = gdf.copy()
    if col in gdf.columns:
        filled = gdf[col].fillna(encoder.unknown_label)
        gdf[f"{col}_encoded"] = encoder.transform(filled)
    else:
        print(f"  [WARN] Column '{col}' not in GeoDataFrame — skipping encoding")
        gdf[f"{col}_encoded"] = 0  # all unknown
    return gdf


# Apply encoding to all cities
for city in CONFIG["split_map"]:
    bgt_data[city] = apply_encoding(bgt_data[city], bgt_col, bgt_encoder)
    brt_data[city] = apply_encoding(brt_data[city], brt_col, brt_encoder)
    print(f"Encoded {city}")

print("\nEncoding complete.")

## 5. Tile each city into fixed-size grid cells

In [ ]:
def make_tile_grid(gdf: gpd.GeoDataFrame, tile_size: float, crs: str) -> gpd.GeoDataFrame:
    """
    Create a regular grid of square tiles covering the bounding box of gdf.
    Returns a GeoDataFrame where each row is one tile polygon.
    """
    minx, miny, maxx, maxy = gdf.total_bounds

    xs = np.arange(minx, maxx, tile_size)
    ys = np.arange(miny, maxy, tile_size)

    tiles = []
    for x in xs:
        for y in ys:
            tile_geom = box(x, y, x + tile_size, y + tile_size)
            tiles.append({"geometry": tile_geom,
                          "tile_x": x, "tile_y": y})

    grid = gpd.GeoDataFrame(tiles, crs=crs)
    grid["tile_id"] = range(len(grid))
    return grid


def filter_tiles_by_coverage(grid: gpd.GeoDataFrame,
                              data: gpd.GeoDataFrame,
                              min_coverage: float) -> gpd.GeoDataFrame:
    """
    Remove tiles where the fraction of the tile covered by data
    is below min_coverage. Avoids empty edge tiles.
    """
    tile_area = grid.geometry.iloc[0].area
    clipped = gpd.clip(data, grid)

    # Area of data within each tile
    joined = gpd.sjoin(grid, clipped[["geometry"]], how="left", predicate="intersects")
    data_area = (
        joined.groupby("tile_id")
        .apply(lambda g: gpd.GeoDataFrame(g, crs=grid.crs).geometry.area.sum())
        .rename("data_area")
    )

    grid = grid.join(data_area, on="tile_id")
    grid["coverage"] = grid["data_area"].fillna(0) / tile_area
    return grid[grid["coverage"] >= min_coverage].copy()


def clip_to_tile(gdf: gpd.GeoDataFrame, tile_geom) -> gpd.GeoDataFrame:
    """Clip a GeoDataFrame to a single tile polygon."""
    clipped = gpd.clip(gdf, tile_geom)
    return clipped.reset_index(drop=True)


print("Tiling functions defined.")

In [ ]:
# Build tiles and save BGT+BRT pairs per tile per city

tile_summary = {}

for city, split in CONFIG["split_map"].items():
    print(f"\nTiling {city.upper()} → {split}")

    bgt = bgt_data[city]
    brt = brt_data[city]

    # Build grid from BGT extent
    grid = make_tile_grid(bgt, CONFIG["tile_size_m"], CONFIG["crs"])
    print(f"  Initial grid: {len(grid)} tiles")

    # Filter low-coverage tiles
    grid = filter_tiles_by_coverage(grid, bgt, CONFIG["tile_min_coverage"])
    print(f"  After coverage filter: {len(grid)} tiles")

    out_dir = CONFIG["output_root"] / split / city
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = 0
    for _, tile_row in grid.iterrows():
        tile_id = tile_row["tile_id"]
        tile_geom = tile_row["geometry"]

        bgt_tile = clip_to_tile(bgt, tile_geom)
        brt_tile = clip_to_tile(brt, tile_geom)

        # Skip tiles where BGT or BRT is empty after clipping
        if len(bgt_tile) == 0 or len(brt_tile) == 0:
            continue

        # Save as GeoJSON (readable and portable)
        bgt_tile.to_file(out_dir / f"tile_{tile_id:04d}_bgt.geojson", driver="GeoJSON")
        brt_tile.to_file(out_dir / f"tile_{tile_id:04d}_brt.geojson", driver="GeoJSON")
        saved += 1

    tile_summary[city] = {"split": split, "tiles_saved": saved}
    print(f"  Saved {saved} tile pairs to {out_dir}")

print("\n── Tile summary ──")
for city, info in tile_summary.items():
    print(f"  {city:12s} [{info['split']:5s}]  {info['tiles_saved']} tile pairs")

## 6. Save tile index

A simple JSON index listing all tile paths — used by later notebooks.

In [ ]:
index = {"train": [], "val": [], "test": []}

for city, split in CONFIG["split_map"].items():
    city_dir = CONFIG["output_root"] / split / city
    bgt_files = sorted(city_dir.glob("*_bgt.geojson"))

    for bgt_path in bgt_files:
        brt_path = Path(str(bgt_path).replace("_bgt.geojson", "_brt.geojson"))
        if brt_path.exists():
            index[split].append({
                "city": city,
                "bgt": str(bgt_path),
                "brt": str(brt_path),
            })

index_path = CONFIG["output_root"] / "tile_index.json"
with open(index_path, "w") as f:
    json.dump(index, f, indent=2)

print(f"Tile index saved to {index_path}")
print(f"  train: {len(index['train'])} tiles")
print(f"  val  : {len(index['val'])} tiles")
print(f"  test : {len(index['test'])} tiles")

## 7. Quick sanity plot

In [ ]:
# Plot the first tile of each split to verify alignment
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (split, tiles) in zip(axes, index.items()):
    if not tiles:
        ax.set_title(f"{split} — no tiles")
        continue

    first = tiles[0]
    bgt_tile = gpd.read_file(first["bgt"])
    brt_tile = gpd.read_file(first["brt"])

    bgt_tile.plot(ax=ax, color="steelblue", alpha=0.5, label="BGT")
    brt_tile.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=1.2, label="BRT")

    ax.set_title(f"{split.upper()} — {first['city']} tile 0")
    ax.legend(loc="lower right", fontsize=8)
    ax.set_axis_off()

plt.suptitle("BGT (blue fill) vs BRT (red outline) — first tile per split", y=1.02)
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "sanity_check.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sanity plot saved.")